# CASMI 2026 · Yash · Model 1: analog + modification enumeration (E711, gate 0.7)

Generates new candidate structures (Class 3) by applying one small edit (+/-O, +/-CH2, +/-H2, +/-hexose) to library molecules whose mass is exactly one edit away from the query. Library candidates keep the baseline cosine; generated products get their parent's cosine, scaled to 0.3 when the best library match already has cosine >= 0.7.

**Inputs:** competition data only (the shared code is embedded in the first cell). **Settings:** Internet OFF, CPU, no weights.

In [ ]:
import glob, os, subprocess, sys, time
t0 = time.time()

# Shared team code (src/ @ f93e677) and Yash's models, embedded below. No team dataset needed.
REPO = '/kaggle/working/casmi_repo'
FILES = {
    'src/__init__.py': '',
    'src/data/__init__.py': '',
    'src/data/build_reference.py': '"""Build a compact spectral reference library from train.parquet.\n\nOutputs (in --out, default data/processed/):\n  ref_meta.parquet   one row per kept spectrum: struct_idx, ion_mode, adduct, lib, peak offsets\n  ref_peaks.npz      concatenated cleaned peaks (mz, intensity) for all kept spectra\n  structures.parquet one row per inchikey14: smiles, formula, neutral mass, n_spectra\n\nUsage:  python -m src.data.build_reference --train data/raw/train.parquet --out data/processed\n"""\nimport argparse\nimport os\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\n\nfrom src.data.spectra import clean_peaks, formula_mass\n\nCOLS = ["ingest_lib", "normalized_smiles", "inchikey14", "molecular_formula", "ionization_mode",\n        "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"]\n\n\ndef build(train_path, out_dir, top_n=64, min_rel=0.01, min_peaks=3):\n    os.makedirs(out_dir, exist_ok=True)\n    pf = pq.ParquetFile(train_path)\n    metas, mz_chunks, int_chunks = [], [], []\n    offset, t0 = 0, time.time()\n    for rg in range(pf.metadata.num_row_groups):\n        tbl = pf.read_row_group(rg, columns=COLS)\n        mz_list = tbl.column("ms2_mzs").to_pylist()\n        in_list = tbl.column("ms2_normalized_intensities").to_pylist()\n        df = tbl.drop(["ms2_mzs", "ms2_normalized_intensities"]).to_pandas()\n        starts, lens = np.zeros(len(df), np.int64), np.zeros(len(df), np.int32)\n        for i, (mz, it, pmz) in enumerate(zip(mz_list, in_list, df.precursor_mz.values)):\n            m, s = clean_peaks(mz or [], it or [], pmz, top_n=top_n, min_rel=min_rel)\n            starts[i], lens[i] = offset, m.size\n            if m.size:\n                mz_chunks.append(m)\n                int_chunks.append(s)\n                offset += m.size\n        df["peak_start"], df["n_peaks"] = starts, lens\n        metas.append(df[df.n_peaks >= min_peaks])\n        print(f"row group {rg + 1}/{pf.metadata.num_row_groups}  {time.time() - t0:.0f}s", flush=True)\n        del tbl, mz_list, in_list\n\n    meta = pd.concat(metas, ignore_index=True)\n    mz_all = np.concatenate(mz_chunks)\n    int_all = np.concatenate(int_chunks)\n\n    structs = (meta.groupby("inchikey14")\n               .agg(smiles=("normalized_smiles", "first"),\n                    formula=("molecular_formula", "first"),\n                    n_spectra=("normalized_smiles", "size"),\n                    n_libs=("ingest_lib", "nunique"))\n               .reset_index())\n    structs["mass"] = structs.formula.map(formula_mass)\n    structs = structs.dropna(subset=["mass"]).sort_values("mass").reset_index(drop=True)\n    idx = pd.Series(np.arange(len(structs)), index=structs.inchikey14)\n    meta["struct_idx"] = meta.inchikey14.map(idx)\n    meta = meta.dropna(subset=["struct_idx"])\n    meta["struct_idx"] = meta.struct_idx.astype(np.int32)\n    meta = meta[["struct_idx", "inchikey14", "ingest_lib", "ionization_mode", "adduct",\n                 "precursor_mz", "peak_start", "n_peaks"]]\n\n    meta.to_parquet(os.path.join(out_dir, "ref_meta.parquet"), index=False)\n    structs.to_parquet(os.path.join(out_dir, "structures.parquet"), index=False)\n    np.savez(os.path.join(out_dir, "ref_peaks.npz"), mz=mz_all, intensity=int_all)\n    print(f"done: {len(meta):,} spectra, {len(structs):,} structures, {mz_all.size:,} peaks, "\n          f"{time.time() - t0:.0f}s")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--train", default="data/raw/train.parquet")\n    ap.add_argument("--out", default="data/processed")\n    ap.add_argument("--top_n", type=int, default=64)\n    args = ap.parse_args()\n    build(args.train, args.out, top_n=args.top_n)\n',
    'src/data/dataset.py': '"""Load cleaned training spectra for model training, without leaking validation molecules.\n\nExample:\n    from src.data.dataset import load_training\n    df = load_training(max_per_structure=3)          # ~740k spectra, ~30 s, ~5 GB RAM\n    df = load_training(libs=["gnps", "riken"], sample=50_000)   # quick experiments\n\nReturned columns: inchikey14, smiles, formula, ingest_lib, adduct, ionization_mode,\nprecursor_mz, collision_energy (mean eV or NaN), mz (float32 array), intensity (float32 array).\nPeaks are cleaned with src.data.spectra.clean_peaks (1% floor, top-N, sqrt, L2-normalised).\n"""\nimport numpy as np\nimport pandas as pd\nimport pyarrow.compute as pc\nimport pyarrow.parquet as pq\n\nfrom src.data.spectra import ADDUCT_DELTA, clean_peaks\n\nTRAIN_PATH = "data/raw/train.parquet"\nVAL_LIB = "enveda-np-examples"\nTEST_ADDUCTS = list(ADDUCT_DELTA)\nCOLS = ["ingest_lib", "normalized_smiles", "inchikey14", "molecular_formula", "ionization_mode",\n        "adduct", "precursor_mz", "precursor_error_ppm", "collision_energy_ev",\n        "ms2_mzs", "ms2_normalized_intensities"]\n\n\ndef validation_keys(train_path=TRAIN_PATH):\n    """InChIKey14s of the validation molecules. Never train on these."""\n    t = pq.read_table(train_path, columns=["ingest_lib", "inchikey14"],\n                      filters=[("ingest_lib", "=", VAL_LIB)])\n    return set(t.column("inchikey14").to_pylist())\n\n\ndef _mean_ce(x):\n    return float(np.mean(x)) if x is not None and len(x) else np.nan\n\n\ndef load_training(train_path=TRAIN_PATH, libs=None, exclude_val=True, max_ppm=20.0,\n                  adducts=TEST_ADDUCTS, max_per_structure=None, sample=None, top_n=64,\n                  min_peaks=3, seed=0):\n    """Load cleaned training spectra.\n\n    libs:              list of ingest_lib values to keep (None = all except the validation library)\n    exclude_val:       drop every spectrum whose structure is a validation molecule (no leakage)\n    max_ppm:           drop spectra with |precursor_error_ppm| above this (likely mislabelled)\n    adducts:           keep only these adducts (default: the 10 that occur in the test set)\n    max_per_structure: keep at most this many spectra per structure (saves memory)\n    sample:            randomly keep this many spectra in total (for quick experiments)\n    """\n    rng = np.random.default_rng(seed)\n    meta_cols = [c for c in COLS if c not in ("ms2_mzs", "ms2_normalized_intensities")]\n    meta = pq.read_table(train_path, columns=meta_cols).to_pandas()\n    meta["row"] = np.arange(len(meta))\n\n    keep = meta.ingest_lib != VAL_LIB\n    if libs is not None:\n        keep &= meta.ingest_lib.isin(libs)\n    if exclude_val:\n        keep &= ~meta.inchikey14.isin(validation_keys(train_path))\n    if max_ppm is not None:\n        keep &= meta.precursor_error_ppm.abs().fillna(0) <= max_ppm\n    if adducts is not None:\n        keep &= meta.adduct.isin(adducts)\n    meta = meta[keep]\n    if max_per_structure:\n        meta = meta.sample(frac=1, random_state=seed).groupby("inchikey14").head(max_per_structure)\n    if sample and sample < len(meta):\n        meta = meta.iloc[rng.choice(len(meta), sample, replace=False)]\n    meta = meta.sort_values("row")\n\n    # Read peaks only for the kept rows, one row group at a time.\n    pf = pq.ParquetFile(train_path)\n    wanted = meta.row.values\n    mz_out, in_out, start = [], [], 0\n    for rg in range(pf.metadata.num_row_groups):\n        n = pf.metadata.row_group(rg).num_rows\n        lo, hi = np.searchsorted(wanted, [start, start + n])\n        if hi > lo:\n            tbl = pf.read_row_group(rg, columns=["ms2_mzs", "ms2_normalized_intensities"])\n            local = wanted[lo:hi] - start\n            tbl = tbl.take(local)\n            for mz, it, pmz in zip(tbl.column(0).to_pylist(), tbl.column(1).to_pylist(),\n                                   meta.precursor_mz.values[lo:hi]):\n                m, s = clean_peaks(mz or [], it or [], pmz, top_n=top_n)\n                mz_out.append(m)\n                in_out.append(s)\n        start += n\n\n    df = pd.DataFrame({\n        "inchikey14": meta.inchikey14.values,\n        "smiles": meta.normalized_smiles.values,\n        "formula": meta.molecular_formula.values,\n        "ingest_lib": meta.ingest_lib.values,\n        "adduct": meta.adduct.values,\n        "ionization_mode": meta.ionization_mode.values,\n        "precursor_mz": meta.precursor_mz.values,\n        "collision_energy": meta.collision_energy_ev.map(_mean_ce).values,\n        "mz": mz_out,\n        "intensity": in_out,\n    })\n    df = df[df.mz.map(len) >= min_peaks].reset_index(drop=True)\n    return df\n\n\ndef split_by_structure(df, val_frac=0.1, seed=0):\n    """Train/val split where no structure (inchikey14) appears on both sides."""\n    keys = df.inchikey14.unique()\n    rng = np.random.default_rng(seed)\n    val_keys = set(rng.choice(keys, int(len(keys) * val_frac), replace=False))\n    is_val = df.inchikey14.isin(val_keys)\n    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)\n',
    'src/data/spectra.py': '"""Shared spectrum utilities: adduct masses, neutral mass, peak cleaning."""\nimport re\n\nimport numpy as np\n\nPROTON = 1.007276\n\n# Mass added to the neutral molecule M to obtain the singly charged ion m/z.\nADDUCT_DELTA = {\n    "[M+H]+": PROTON,\n    "[M+NH4]+": 18.033823,\n    "[M-H2O+H]+": PROTON - 18.010565,\n    "[M-2H2O+H]+": PROTON - 2 * 18.010565,\n    "[M+Na]+": 22.989218,\n    "[M+K]+": 38.963158,\n    "[M-H]-": -PROTON,\n    "[M-H2O-H]-": -PROTON - 18.010565,\n    "[M+CH2O2-H]-": 44.998201,\n    "[M+Cl]-": 34.969402,\n}\n\nMONO = {\n    "H": 1.0078250319, "C": 12.0, "N": 14.0030740052, "O": 15.9949146221,\n    "P": 30.97376151, "S": 31.97207069, "F": 18.99840320, "Cl": 34.96885271,\n    "Br": 78.9183376, "I": 126.904468, "Si": 27.9769265327, "B": 11.0093055,\n    "Se": 79.9165218, "Na": 22.98976966, "K": 38.9637069, "As": 74.9215964,\n}\n_FORMULA_RE = re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(formula):\n    """Monoisotopic neutral mass of a molecular formula, NaN if unparseable/charged."""\n    if not isinstance(formula, str) or any(c in formula for c in "+-[]()."):\n        return np.nan\n    total = 0.0\n    for el, n in _FORMULA_RE.findall(formula):\n        if el not in MONO:\n            return np.nan\n        total += MONO[el] * (int(n) if n else 1)\n    return total\n\n\ndef neutral_mass(precursor_mz, adduct):\n    """Neutral monoisotopic mass implied by a precursor m/z and adduct (NaN if adduct unknown)."""\n    delta = ADDUCT_DELTA.get(adduct)\n    return np.nan if delta is None else precursor_mz - delta\n\n\ndef clean_peaks(mzs, ints, precursor_mz, top_n=64, min_rel=0.01, max_above_precursor=2.0):\n    """Relative-intensity floor, drop peaks above precursor, keep top-N, sqrt-scale, L2-normalise.\n\n    Returns (mz float32, intensity float32) sorted by m/z; empty arrays if nothing survives.\n    """\n    mzs = np.asarray(mzs, dtype=np.float64)\n    ints = np.asarray(ints, dtype=np.float64)\n    if mzs.size == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32)\n    keep = ints >= min_rel * ints.max()\n    if precursor_mz and np.isfinite(precursor_mz):\n        keep &= mzs <= precursor_mz + max_above_precursor\n    mzs, ints = mzs[keep], ints[keep]\n    if mzs.size > top_n:\n        idx = np.argpartition(-ints, top_n)[:top_n]\n        mzs, ints = mzs[idx], ints[idx]\n    order = np.argsort(mzs)\n    mzs, ints = mzs[order], np.sqrt(ints[order])\n    norm = np.linalg.norm(ints)\n    if norm > 0:\n        ints = ints / norm\n    return mzs.astype(np.float32), ints.astype(np.float32)\n\n\ndef cosine(mz_a, int_a, mz_b, int_b, tol=0.01):\n    """Greedy cosine similarity between two cleaned (L2-normalised) spectra."""\n    if mz_a.size == 0 or mz_b.size == 0:\n        return 0.0\n    diff = np.abs(mz_a[:, None] - mz_b[None, :])\n    ia, ib = np.nonzero(diff <= tol)\n    if ia.size == 0:\n        return 0.0\n    prod = int_a[ia] * int_b[ib]\n    order = np.argsort(-prod)\n    used_a, used_b, score = set(), set(), 0.0\n    for k in order:\n        a, b = ia[k], ib[k]\n        if a not in used_a and b not in used_b:\n            used_a.add(a)\n            used_b.add(b)\n            score += prod[k]\n    return float(score)\n',
    'src/evaluation/__init__.py': '',
    'src/evaluation/metric.py': '"""Local re-implementation of the competition metric (MRR@25 on InChIKey14).\n\nOfficial notebook: https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank\nThe official metric tautomer-canonicalises with RDKit 2026.03.3 before taking the InChIKey14.\n"""\nfrom functools import lru_cache\n\nimport numpy as np\n\nK = 25\n\n\n@lru_cache(maxsize=None)\ndef smiles_to_ik14(smiles, tautomer=True):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n\n    RDLogger.DisableLog("rdApp.*")\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None:\n        return None\n    if tautomer:\n        try:\n            mol = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)\n        except Exception:\n            pass\n    key = Chem.MolToInchiKey(mol)\n    return key.split("-")[0] if key else None\n\n\ndef mrr_from_keys(pred_keys, true_keys, k=K):\n    """pred_keys: list of ranked key lists; true_keys: list of keys. Returns MRR@k."""\n    scores = []\n    for preds, truth in zip(pred_keys, true_keys):\n        rr = 0.0\n        for rank, key in enumerate(preds[:k], start=1):\n            if key == truth:\n                rr = 1.0 / rank\n                break\n        scores.append(rr)\n    return float(np.mean(scores)) if scores else 0.0\n\n\ndef mrr_from_smiles(pred_smiles, true_smiles, k=K):\n    """Same as the official metric: SMILES lists → tautomer-canonical InChIKey14 → MRR@k."""\n    pred_keys = [[smiles_to_ik14(s) for s in preds[:k]] for preds in pred_smiles]\n    true_keys = [smiles_to_ik14(s) for s in true_smiles]\n    return mrr_from_keys(pred_keys, true_keys, k)\n',
    'src/evaluation/validate.py': '"""Validate any Scorer on the team\'s shared validation set, and build Kaggle submissions.\n\nEveryone uses the same protocol so our numbers are comparable in the report:\n  * queries   = the 250 molecules of `enveda-np-examples` (same instrument as the hidden test)\n  * reference = every other library (enveda-np-examples removed)\n  * candidates = reference structures within ±15 ppm of the molecule\'s neutral mass\n  * metric    = MRR@25 on InChIKey14 (same as Kaggle)\n\n    from src.evaluation.validate import evaluate\n    result = evaluate(MyModel())\n    print(result["mrr"])\n\nCommand line (baseline check, should print MRR@25 ≈ 0.879):\n    python -m src.evaluation.validate\n"""\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.parquet as pq\n\nfrom src.evaluation.metric import mrr_from_keys, smiles_to_ik14\nfrom src.models.library_search import ReferenceLibrary, candidate_table, prepare_queries\n\nVAL_LIB = "enveda-np-examples"\nK = 25\n_CACHE = {}\n\n\ndef _add_extra(cands, extra_smiles):\n    if not extra_smiles:\n        return cands\n    seen = set(cands.inchikey14)\n    rows = []\n    for s in extra_smiles:\n        key = smiles_to_ik14(s, tautomer=False)\n        if key and key not in seen:\n            seen.add(key)\n            rows.append({"struct_idx": -1, "smiles": s, "inchikey14": key, "formula": None,\n                         "mass": np.nan, "n_spectra": 0, "cosine": 0.0})\n    return pd.concat([cands, pd.DataFrame(rows)], ignore_index=True) if rows else cands\n\n\ndef rank(scorer, spectra, cands):\n    """Score candidates and return them sorted best first (ties → more library spectra)."""\n    cands = _add_extra(cands, scorer.extra_candidates(spectra))\n    if len(cands) == 0:\n        return cands\n    scores = np.asarray(scorer.score(spectra, cands), dtype=float)\n    assert len(scores) == len(cands), f"{scorer.name}: got {len(scores)} scores for {len(cands)} candidates"\n    cands = cands.assign(score=scores)\n    return cands.sort_values(["score", "n_spectra"], ascending=False, kind="stable")\n\n\ndef load_validation(ref_dir="data/processed", train_path="data/raw/train.parquet", ppm=15):\n    """Validation queries + their candidate tables (cached, ~5 s the first time)."""\n    key = (ref_dir, ppm)\n    if key not in _CACHE:\n        val = pq.read_table(train_path, filters=[("ingest_lib", "=", VAL_LIB)]).to_pandas()\n        val = val[val.adduct.isin(["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+",\n                                   "[M+K]+", "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"])]\n        val["molecule_id"] = val.inchikey14\n        queries = prepare_queries(val)\n        lib = ReferenceLibrary(ref_dir, exclude_libs=[VAL_LIB])\n        cands = {m: candidate_table(lib, sp, ppm=ppm) for m, sp in queries.items()}\n        _CACHE[key] = (queries, cands)\n    return _CACHE[key]\n\n\ndef evaluate(scorer, ref_dir="data/processed", ppm=15, hide_true=False, verbose=True, limit=None):\n    """Returns dict(mrr, top1, top10, top25, per_molecule DataFrame).\n\n    hide_true=True removes the correct structure from the library candidates, which simulates\n    Class 2/3 test molecules (no reference spectra). Then the only way to score is for the model\'s\n    extra_candidates() to propose the right structure. Use this for candidate-expansion and de novo models.\n    limit: evaluate only the first N molecules (quick debugging).\n    """\n    t0 = time.time()\n    queries, cands = load_validation(ref_dir, ppm=ppm)\n    rows = []\n    for i, (mol, spectra) in enumerate(queries.items()):\n        if limit and i >= limit:\n            break\n        c = cands[mol]\n        if hide_true:\n            c = c[c.inchikey14 != mol]\n        ranked = rank(scorer, spectra, c)\n        keys = list(ranked.inchikey14[:K])\n        pos = keys.index(mol) + 1 if mol in keys else None\n        rows.append({"molecule": mol, "rank": pos, "n_candidates": len(ranked),\n                     "rr": 1.0 / pos if pos else 0.0})\n    per = pd.DataFrame(rows)\n    res = {\n        "model": scorer.name,\n        "mrr": per.rr.mean(),\n        "top1": (per["rank"] == 1).mean(),\n        "top10": (per["rank"] <= 10).mean(),\n        "top25": per["rank"].notna().mean(),\n        "per_molecule": per,\n    }\n    if verbose:\n        tag = " (true structure hidden)" if hide_true else ""\n        print(f"[{scorer.name}{tag}] MRR@25 = {res[\'mrr\']:.4f} | top-1 {res[\'top1\']:.3f} | "\n              f"top-10 {res[\'top10\']:.3f} | top-25 {res[\'top25\']:.3f} | "\n              f"{len(per)} molecules | {time.time() - t0:.0f}s")\n    return res\n\n\ndef make_submission(scorer, test_path, ref_dir, out_path="submission.csv", ppm=15):\n    """Run a scorer on a test-format parquet and write a valid submission.csv."""\n    test = pd.read_parquet(test_path)\n    lib = ReferenceLibrary(ref_dir)\n    queries = prepare_queries(test)\n    rows = []\n    for mol in test.molecule_id.drop_duplicates():\n        ranked = rank(scorer, queries[mol], candidate_table(lib, queries[mol], ppm=ppm))\n        smiles = list(ranked.drop_duplicates("inchikey14").smiles[:K]) or ["CCO"]\n        rows.append({"molecule_id": mol, "smiles": ";".join(smiles)})\n    sub = pd.DataFrame(rows)\n    sub.to_csv(out_path, index=False)\n    return sub\n\n\nif __name__ == "__main__":\n    from src.models.base import CosineScorer, MassOnlyScorer\n    evaluate(MassOnlyScorer())\n    evaluate(CosineScorer())\n',
    'src/features/__init__.py': '',
    'src/features/featurize.py': '"""Featurisers shared by all models: binned spectra, peak tensors, molecular fingerprints."""\nfrom functools import lru_cache\n\nimport numpy as np\n\nADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",\n           "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]\n\n\n# ---------- spectrum features ----------\n\ndef bin_spectrum(mz, intensity, max_mz=1000.0, bin_size=1.0):\n    """Fixed-length vector: sum of intensities per m/z bin. Good input for MLP/CNN models."""\n    n = int(np.ceil(max_mz / bin_size))\n    v = np.zeros(n, np.float32)\n    idx = (np.asarray(mz) / bin_size).astype(int)\n    ok = (idx >= 0) & (idx < n)\n    np.add.at(v, idx[ok], np.asarray(intensity)[ok])\n    return v\n\n\ndef peak_tensor(mz, intensity, precursor_mz, max_peaks=64):\n    """(max_peaks, 3) array of [m/z, intensity, neutral loss] plus a padding mask.\n    Good input for transformer / set models. Row 0 is reserved for the precursor."""\n    x = np.zeros((max_peaks, 3), np.float32)\n    mask = np.zeros(max_peaks, bool)\n    x[0] = [precursor_mz, 1.0, 0.0]\n    mask[0] = True\n    mz = np.asarray(mz)[: max_peaks - 1]\n    it = np.asarray(intensity)[: max_peaks - 1]\n    k = len(mz)\n    x[1:k + 1, 0] = mz\n    x[1:k + 1, 1] = it\n    x[1:k + 1, 2] = precursor_mz - mz\n    mask[1:k + 1] = True\n    return x, mask\n\n\ndef metadata_vector(adduct, collision_energy, precursor_mz):\n    """One-hot adduct + scaled collision energy + scaled precursor m/z (12 values)."""\n    v = np.zeros(len(ADDUCTS) + 2, np.float32)\n    if adduct in ADDUCTS:\n        v[ADDUCTS.index(adduct)] = 1.0\n    v[-2] = 0.0 if collision_energy is None or not np.isfinite(collision_energy) else collision_energy / 100.0\n    v[-1] = precursor_mz / 1000.0\n    return v\n\n\n# ---------- molecule features ----------\n\n@lru_cache(maxsize=500_000)\ndef _fp_bytes(smiles, n_bits, radius):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n\n    RDLogger.DisableLog("rdApp.*")\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None:\n        return None\n    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)\n    return gen.GetFingerprintAsNumPy(mol).astype(np.uint8).tobytes()\n\n\ndef morgan_fp(smiles, n_bits=2048, radius=2):\n    """Morgan (ECFP4) bit vector as uint8 array of 0/1, or all zeros if SMILES is invalid."""\n    b = _fp_bytes(smiles, n_bits, radius)\n    return np.zeros(n_bits, np.uint8) if b is None else np.frombuffer(b, np.uint8).copy()\n\n\ndef morgan_fps(smiles_list, n_bits=2048, radius=2):\n    return np.stack([morgan_fp(s, n_bits, radius) for s in smiles_list]) if len(smiles_list) else \\\n        np.zeros((0, n_bits), np.uint8)\n\n\ndef tanimoto(query, fps):\n    """Tanimoto between one fingerprint (bits or probabilities) and a matrix of bit fingerprints."""\n    q = np.asarray(query, np.float32)\n    F = np.asarray(fps, np.float32)\n    inter = F @ q\n    union = F.sum(1) + q.sum() - inter\n    return np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0)\n\n\ndef cosine_sim(query, fps):\n    q = np.asarray(query, np.float32)\n    F = np.asarray(fps, np.float32)\n    return (F @ q) / (np.linalg.norm(F, axis=1) * np.linalg.norm(q) + 1e-9)\n',
    'src/models/__init__.py': '',
    'src/models/base.py': '"""The one interface every team model implements.\n\nA model is a *scorer*: given one molecule\'s spectra and a table of candidate structures,\nit returns one score per candidate (higher = more likely). The shared pipeline handles\neverything else: candidate retrieval by precursor mass, ranking, validation and the\nKaggle submission file.\n\n    class MyModel(Scorer):\n        name = "fingerprint-mlp"\n        def score(self, spectra, candidates):\n            ...  # return np.ndarray of len(candidates)\n\n`spectra`: list of dicts, one per spectrum of the molecule, with keys\n    mz, intensity (cleaned float32 arrays), precursor_mz, adduct, ionization_mode, collision_energy\n`candidates`: pandas DataFrame with columns\n    smiles, inchikey14, formula, mass, n_spectra (times seen in libraries),\n    cosine (baseline library-search score; useful as a feature)\n\nOptional: override `extra_candidates(spectra)` to return additional SMILES that are not in the\ntraining library (e.g. from PubChem/COCONUT or a generative model). They are added to\n`candidates` (with n_spectra=0 and cosine=0) before `score` is called.\n"""\nimport numpy as np\n\n\nclass Scorer:\n    name = "scorer"\n\n    def score(self, spectra, candidates):\n        raise NotImplementedError\n\n    def extra_candidates(self, spectra):\n        return []\n\n\nclass CosineScorer(Scorer):\n    """The baseline (E002): library-search cosine."""\n    name = "library-cosine"\n\n    def score(self, spectra, candidates):\n        return candidates.cosine.values\n\n\nclass MassOnlyScorer(Scorer):\n    """E001: mass match only, prefer frequently measured structures."""\n    name = "mass-only"\n\n    def score(self, spectra, candidates):\n        return np.log1p(candidates.n_spectra.values.astype(float))\n',
    'src/models/library_search.py': '"""Baseline: precursor-mass candidate retrieval + spectral library search.\n\nFor each query molecule:\n  1. neutral mass from each spectrum\'s precursor m/z and adduct (median across spectra)\n  2. candidates = reference structures whose neutral mass is within `ppm`\n  3. score each candidate = mean over query spectra of the best cosine against the\n     candidate\'s reference spectra (same ionisation mode)\n  4. rank by score, ties broken by how often the structure appears in the libraries\n"""\nimport os\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.data.spectra import clean_peaks, cosine, neutral_mass\n\nK = 25\n\n\nclass ReferenceLibrary:\n    def __init__(self, ref_dir, exclude_libs=()):\n        meta = pd.read_parquet(os.path.join(ref_dir, "ref_meta.parquet"))\n        self.structs = pd.read_parquet(os.path.join(ref_dir, "structures.parquet"))\n        peaks = np.load(os.path.join(ref_dir, "ref_peaks.npz"))\n        self.mz, self.inten = peaks["mz"], peaks["intensity"]\n        if exclude_libs:\n            meta = meta[~meta.ingest_lib.isin(exclude_libs)]\n        meta = meta.sort_values("struct_idx").reset_index(drop=True)\n        self.meta = meta\n        n = len(self.structs)\n        counts = np.bincount(meta.struct_idx.values, minlength=n)\n        self.spec_start = np.concatenate([[0], np.cumsum(counts)[:-1]])\n        self.spec_count = counts\n        # Structures with no remaining spectra are not retrievable (mimics unseen molecules).\n        self.available = counts > 0\n        self.masses = self.structs.mass.values\n        self.prior = np.log1p(counts)\n        self.mode = meta.ionization_mode.values\n        self.pstart = meta.peak_start.values\n        self.plen = meta.n_peaks.values\n\n    def candidates(self, mass, ppm):\n        tol = mass * ppm * 1e-6\n        lo, hi = np.searchsorted(self.masses, [mass - tol, mass + tol])\n        idx = np.arange(lo, hi)\n        return idx[self.available[idx]]\n\n    def spectra_of(self, s):\n        a = self.spec_start[s]\n        return range(a, a + self.spec_count[s])\n\n    def peaks(self, j):\n        a = self.pstart[j]\n        return self.mz[a:a + self.plen[j]], self.inten[a:a + self.plen[j]]\n\n\ndef query_molecule(lib, spectra, ppm=15, use_cosine=True, max_ref_per_struct=50):\n    """spectra: list of dicts with precursor_mz, adduct, ionization_mode, mz, intensity (cleaned).\n    Returns list of (struct_idx, score) best first."""\n    masses = [neutral_mass(s["precursor_mz"], s["adduct"]) for s in spectra]\n    masses = [m for m in masses if np.isfinite(m)]\n    if not masses:\n        return []\n    cands = lib.candidates(float(np.median(masses)), ppm)\n    if cands.size == 0:\n        return []\n    scores = np.zeros(cands.size)\n    if use_cosine:\n        for ci, s in enumerate(cands):\n            refs = list(lib.spectra_of(s))[:max_ref_per_struct]\n            per_query = []\n            for q in spectra:\n                best = 0.0\n                for j in refs:\n                    if lib.mode[j] != q["ionization_mode"]:\n                        continue\n                    mz_r, in_r = lib.peaks(j)\n                    best = max(best, cosine(q["mz"], q["intensity"], mz_r, in_r))\n                per_query.append(best)\n            scores[ci] = np.mean(per_query)\n    # tiny prior so ties (and the mass-only baseline) prefer frequently measured structures\n    total = scores + 1e-3 * lib.prior[cands]\n    order = np.argsort(-total)\n    return [(int(cands[i]), float(total[i])) for i in order]\n\n\ndef candidate_table(lib, spectra, ppm=15, with_cosine=True):\n    """All mass-matched candidates for one molecule as a DataFrame:\n    struct_idx, smiles, inchikey14, formula, mass, n_spectra, cosine (baseline spectral score)."""\n    ranked = query_molecule(lib, spectra, ppm=ppm, use_cosine=with_cosine)\n    if not ranked:\n        return pd.DataFrame(columns=["struct_idx", "smiles", "inchikey14", "formula", "mass",\n                                     "n_spectra", "cosine"])\n    idx = np.array([s for s, _ in ranked])\n    score = np.array([v for _, v in ranked]) - 1e-3 * lib.prior[idx]\n    st = lib.structs.iloc[idx]\n    return pd.DataFrame({\n        "struct_idx": idx, "smiles": st.smiles.values, "inchikey14": st.inchikey14.values,\n        "formula": st.formula.values, "mass": st.mass.values,\n        "n_spectra": lib.spec_count[idx], "cosine": score,\n    })\n\n\ndef prepare_queries(df):\n    """Group a test-format dataframe into {molecule_id: [spectrum dicts]} with cleaned peaks."""\n    out = {}\n    for row in df.itertuples(index=False):\n        mz, it = clean_peaks(row.ms2_mzs, row.ms2_normalized_intensities, row.precursor_mz)\n        ce = getattr(row, "collision_energy_ev", None)\n        try:\n            ce = float(np.mean(ce)) if len(ce) else np.nan\n        except TypeError:\n            ce = np.nan\n        out.setdefault(row.molecule_id, []).append(dict(\n            precursor_mz=row.precursor_mz, adduct=row.adduct,\n            ionization_mode=row.ionization_mode, collision_energy=ce, mz=mz, intensity=it))\n    return out\n\n\ndef predict(lib, queries, k=K, **kw):\n    """Returns {molecule_id: [smiles, ...]} (up to k, best first, unique InChIKey14)."""\n    smiles = lib.structs.smiles.values\n    preds = {}\n    for mol_id, spectra in queries.items():\n        ranked = query_molecule(lib, spectra, **kw)[:k]\n        preds[mol_id] = [smiles[s] for s, _ in ranked]\n    return preds\n\n\ndef to_submission(preds, molecule_ids, fallback="CCO"):\n    rows = []\n    for m in molecule_ids:\n        guesses = preds.get(m) or [fallback]\n        rows.append({"molecule_id": m, "smiles": ";".join(guesses[:K])})\n    return pd.DataFrame(rows)\n',
    'src/utils/__init__.py': '',
    'members/YashShevkar30/model_analog_enum.py': '"""Model 1 (Yash): analog + modification enumeration for novel (Class 3) molecules.\n\nIdea in one line: natural products come in families, so an unknown molecule is often a known\nlibrary molecule plus or minus one small chemical edit (+O, +CH2, -CH2, +H2, -H2, +/- a hexose).\n\nFor one query molecule:\n  1. M = neutral mass from the precursor m/z and adduct (median over its spectra).\n  2. For every edit with mass change d: the parent ("analog") must weigh M - d. Look up library\n     structures at M - d (15 ppm), score each by spectral cosine against the query spectra and\n     keep the best few. We never look up mass M itself, so the true structure is never pulled\n     back in from the library (that would fake the hide_true score).\n  3. Apply the edit at every possible site with an RDKit reaction, sanitise, and keep products\n     whose exact mass is within 15 ppm of M.\n  4. Return up to 200 unique products from extra_candidates(). score() gives each product its\n     parent\'s cosine (times gen_weight); library candidates keep the baseline cosine.\n\nRun from the repo root:\n    python -m members.YashShevkar30.model_analog_enum --hide_true          # E701\n    python -m members.YashShevkar30.model_analog_enum                      # Class-1 check\n    python -m members.YashShevkar30.model_analog_enum --hide_true --limit 20   # quick debug\n"""\nimport argparse\nimport json\nimport os\nimport time\n\nimport numpy as np\n\nfrom src.data.spectra import cosine, neutral_mass\nfrom src.evaluation.metric import smiles_to_ik14\nfrom src.models.base import Scorer\n\nVAL_LIB = "enveda-np-examples"\nHEXOSE = "C1OC(CO)C(O)C(O)C1O"\n\n# name -> (mass change product - parent in Da, list of reaction SMARTS parent>>product)\nMODIFICATIONS = {\n    "hydroxylation": (15.994915, ["[cH:1]>>[c:1]O"]),\n    "methylation": (14.015650, ["[OX2H1:1]>>[O:1]C",\n                                "[NX3;H1,H2;!$(N-[!#6;!#1]):1]>>[N:1]C"]),\n    "demethylation": (-14.015650, ["[O:1]-[CH3]>>[O:1]", "[N:1]-[CH3]>>[N:1]"]),\n    "hydrogenation": (2.015650, ["[C:1]=[C:2]>>[C:1]-[C:2]"]),\n    "dehydrogenation": (-2.015650, ["[CX4;!H0:1]-[CX4;!H0:2]>>[C:1]=[C:2]"]),\n    "glycosylation": (162.052824, ["[OX2H1:1]>>[O:1]" + HEXOSE]),\n    "deglycosylation": (-162.052824, [\n        "[*:2]-[O:1]-[CH1;R]1-O-[CH1](-[CH2]-[OH])-[CH1](-[OH])-[CH1](-[OH])-[CH1]1-[OH]>>[*:2]-[O:1]"]),\n}\n\n\ndef query_mass(spectra):\n    masses = [neutral_mass(s["precursor_mz"], s["adduct"]) for s in spectra]\n    masses = [m for m in masses if np.isfinite(m)]\n    return float(np.median(masses)) if masses else np.nan\n\n\ndef spectral_score(lib, spectra, struct_idx, max_ref=20):\n    """Same rule as the baseline: mean over query spectra of the best cosine vs the structure\'s\n    reference spectra in the same ionisation mode."""\n    refs = list(lib.spectra_of(struct_idx))[:max_ref]\n    per_query = []\n    for q in spectra:\n        best = 0.0\n        for j in refs:\n            if lib.mode[j] != q["ionization_mode"]:\n                continue\n            mz_r, in_r = lib.peaks(j)\n            best = max(best, cosine(q["mz"], q["intensity"], mz_r, in_r))\n        per_query.append(best)\n    return float(np.mean(per_query)) if per_query else 0.0\n\n\nclass AnalogEnumScorer(Scorer):\n    name = "analog-enum"\n\n    def __init__(self, ref_dir="data/processed", exclude_libs=(VAL_LIB,), ppm=15.0,\n                 analogs_per_mod=3, max_products=200, gen_weight=1.0, mods=None,\n                 gate=None, gated_weight=0.3):\n        from rdkit import RDLogger\n        from rdkit.Chem import AllChem\n\n        from src.models.library_search import ReferenceLibrary\n\n        RDLogger.DisableLog("rdApp.*")\n        # Validation: exclude the validation library, exactly like src.evaluation.validate does.\n        # Kaggle: pass exclude_libs=() to use every library.\n        self.lib = ReferenceLibrary(ref_dir, exclude_libs=list(exclude_libs))\n        self.ppm = ppm\n        self.analogs_per_mod = analogs_per_mod\n        self.max_products = max_products\n        self.gen_weight = gen_weight\n        # Confidence gate: if the best library candidate already matches well (cosine >= gate),\n        # the molecule is probably in the library, so generated products get gated_weight instead.\n        self.gate = gate\n        self.gated_weight = gated_weight\n        mods = mods or list(MODIFICATIONS)\n        self.rxns = {m: (MODIFICATIONS[m][0], [AllChem.ReactionFromSmarts(s) for s in MODIFICATIONS[m][1]])\n                     for m in mods}\n        self._gen = {}      # inchikey14 -> score for the current molecule\'s products\n        self.stats = []     # one dict per molecule, aligned with evaluate()\'s molecule order\n\n    # ---- generation ----\n    def _apply(self, rxns, parent_smiles):\n        from rdkit import Chem\n        mol = Chem.MolFromSmiles(parent_smiles)\n        if mol is None:\n            return [], 0\n        out, n_raw = [], 0\n        for rx in rxns:\n            for prods in rx.RunReactants((mol,)):\n                n_raw += 1\n                p = Chem.Mol(prods[0])\n                try:\n                    Chem.SanitizeMol(p)\n                except Exception:\n                    continue\n                out.append(p)\n        return out, n_raw\n\n    def extra_candidates(self, spectra):\n        from rdkit import Chem\n        from rdkit.Chem.rdMolDescriptors import CalcExactMolWt\n\n        t0 = time.time()\n        self._gen = {}\n        st = {"n_parents": 0, "n_raw": 0, "n_valid": 0, "n_in_mass": 0, "n_unique": 0}\n        M = query_mass(spectra)\n        if not np.isfinite(M):\n            self.stats.append({**st, "keys": [], "seconds": 0.0})\n            return []\n        best = {}   # inchikey14 -> (score, smiles, mod)\n        for mod, (delta, rxns) in self.rxns.items():\n            parents = self.lib.candidates(M - delta, self.ppm)\n            if parents.size == 0:\n                continue\n            scored = sorted(((spectral_score(self.lib, spectra, s), s) for s in parents), reverse=True)\n            for p_score, s in scored[: self.analogs_per_mod]:\n                st["n_parents"] += 1\n                prods, n_raw = self._apply(rxns, self.lib.structs.smiles.values[s])\n                st["n_raw"] += n_raw\n                st["n_valid"] += len(prods)\n                for p in prods:\n                    if abs(CalcExactMolWt(p) - M) > M * self.ppm * 1e-6:\n                        continue\n                    st["n_in_mass"] += 1\n                    smi = Chem.MolToSmiles(p)\n                    key = smiles_to_ik14(smi, tautomer=False)  # same key validate.py dedupes on\n                    if key and (key not in best or p_score > best[key][0]):\n                        best[key] = (p_score, smi, mod)\n        ranked = sorted(best.items(), key=lambda kv: -kv[1][0])[: self.max_products]\n        self._gen = {k: v[0] for k, v in ranked}\n        st["n_unique"] = len(ranked)\n        self.stats.append({**st, "keys": [k for k, _ in ranked], "mods": [v[2] for _, v in ranked],\n                           "seconds": time.time() - t0})\n        return [v[1] for _, v in ranked]\n\n    # ---- scoring ----\n    def score(self, spectra, candidates):\n        scores = candidates.cosine.values.astype(float).copy()\n        gen = candidates.struct_idx.values == -1\n        w = self.gen_weight\n        if self.gate is not None and (~gen).any() and scores[~gen].max() >= self.gate:\n            w = self.gated_weight\n        scores[gen] = [w * self._gen.get(k, 0.0) for k in candidates.inchikey14.values[gen]]\n        return scores\n\n\ndef summarise(res, model):\n    """Generation statistics for the report, joined to evaluate()\'s per-molecule table."""\n    per = res["per_molecule"].copy()\n    st = model.stats[: len(per)]\n    for col in ["n_parents", "n_raw", "n_valid", "n_in_mass", "n_unique", "seconds"]:\n        per[col] = [s[col] for s in st]\n    per["true_generated"] = [m in set(s["keys"]) for m, s in zip(per.molecule, st)]\n    per["true_mod"] = [s["mods"][s["keys"].index(m)] if m in s["keys"] else None\n                       for m, s in zip(per.molecule, st)]\n    summary = {\n        "model": res["model"], "mrr": res["mrr"], "top1": res["top1"], "top10": res["top10"],\n        "top25": res["top25"], "n_molecules": len(per),\n        "mean_products": float(per.n_unique.mean()),\n        "valid_rate": float(per.n_valid.sum() / max(per.n_raw.sum(), 1)),\n        "in_mass_rate": float(per.n_in_mass.sum() / max(per.n_valid.sum(), 1)),\n        "recall_generated": float(per.true_generated.mean()),\n        "true_by_mod": per.true_mod.value_counts().to_dict(),\n        "sec_per_molecule": float(per.seconds.mean()),\n    }\n    return per, summary\n\n\nif __name__ == "__main__":\n    from src.evaluation.validate import evaluate\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--hide_true", action="store_true")\n    ap.add_argument("--limit", type=int, default=None)\n    ap.add_argument("--gen_weight", type=float, default=1.0)\n    ap.add_argument("--analogs_per_mod", type=int, default=3)\n    ap.add_argument("--gate", type=float, default=None)\n    ap.add_argument("--tag", default=None, help="experiment ID, e.g. E701; saves outputs")\n    args = ap.parse_args()\n\n    np.random.seed(4722)\n    model = AnalogEnumScorer(gen_weight=args.gen_weight, analogs_per_mod=args.analogs_per_mod,\n                             gate=args.gate)\n    res = evaluate(model, hide_true=args.hide_true, limit=args.limit)\n    per, summary = summarise(res, model)\n    summary.update(hide_true=args.hide_true, gen_weight=args.gen_weight,\n                   analogs_per_mod=args.analogs_per_mod, gate=args.gate, limit=args.limit)\n    print(json.dumps(summary, indent=1))\n    if args.tag:\n        out = os.path.join(os.path.dirname(__file__), "outputs")\n        os.makedirs(out, exist_ok=True)\n        per.to_csv(os.path.join(out, f"{args.tag}_per_molecule.csv"), index=False)\n        with open(os.path.join(out, f"{args.tag}_summary.json"), "w") as f:\n            json.dump(summary, f, indent=1)\n',
    'members/YashShevkar30/model_smiles_transformer.py': '"""Model 2 (Yash): spectrum -> SELFIES transformer for novel (Class 3) molecules.\n\nIdea in one line: read the peaks of a spectrum with a transformer encoder and write the molecule\nout token by token with a transformer decoder (SELFIES, so every token string decodes to a valid\nmolecule). Beam search proposes up to 25 structures per spectrum; we keep the ones whose exact mass\nis within 15 ppm of the query mass M and hand them to the pipeline through extra_candidates().\n\n  encoder input : one token per peak (Fourier features of m/z and neutral loss, plus intensity),\n                  plus one "meta" token (adduct one-hot, collision energy, precursor m/z, and\n                  Fourier features of the neutral mass M).\n  decoder       : 4 layers, causal self-attention + cross-attention to the peak tokens,\n                  SELFIES tokens (stereo removed first: the metric ignores stereochemistry).\n  training      : teacher forcing on load_training(max_per_structure=2). Validation molecules are\n                  never in the training data (load_training removes them); early stopping uses a\n                  structure-disjoint split of the training data.\n  scoring       : library candidates keep the baseline cosine. A generated structure gets the mean\n                  over the molecule\'s spectra of exp(mean token log-prob) of its best beam (0 for a\n                  spectrum that did not produce it), times gen_weight.\n\nRun from the repo root:\n    python -m members.YashShevkar30.model_smiles_transformer train --sample 20000 --epochs 1 \\\n        --out members/YashShevkar30/weights/smoke.pt                                   # smoke test\n    python -m members.YashShevkar30.model_smiles_transformer evaluate \\\n        --weights members/YashShevkar30/weights/model.pt --hide_true --tag E702        # E702\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom src.data.spectra import neutral_mass\nfrom src.evaluation.metric import smiles_to_ik14\nfrom src.features.featurize import metadata_vector, peak_tensor\nfrom src.models.base import Scorer\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nSEED = 4722\nPAD, BOS, EOS, UNK = 0, 1, 2, 3\nSPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]\nMAX_PEAKS = 64      # peak_tensor rows (row 0 = precursor)\nMAX_TOK = 96        # sequence length including <bos> and <eos>\n\n\ndef pick_device():\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\n# ---------------------------------------------------------------- tokenising structures\ndef smiles_to_tokens(smiles):\n    """SMILES -> list of SELFIES tokens (no stereo), or None if RDKit/SELFIES cannot handle it."""\n    import selfies as sf\n    from rdkit import Chem\n    try:\n        mol = Chem.MolFromSmiles(smiles)\n        if mol is None:\n            return None\n        flat = Chem.MolToSmiles(mol, isomericSmiles=False)\n        return list(sf.split_selfies(sf.encoder(flat)))\n    except Exception:\n        return None\n\n\ndef tokens_to_smiles(tokens):\n    """List of SELFIES tokens -> canonical SMILES (RDKit), or None if invalid."""\n    import selfies as sf\n    from rdkit import Chem\n    try:\n        smi = sf.decoder("".join(tokens))\n        mol = Chem.MolFromSmiles(smi)\n        return None if mol is None else Chem.MolToSmiles(mol)\n    except Exception:\n        return None\n\n\ndef tokenize_structures(df, max_tok=MAX_TOK):\n    """One tokenisation per structure. Returns {inchikey14: tokens} (too-long / failed ones dropped)."""\n    out = {}\n    uniq = df.drop_duplicates("inchikey14")[["inchikey14", "smiles"]]\n    for key, smi in zip(uniq.inchikey14.values, uniq.smiles.values):\n        t = smiles_to_tokens(smi)\n        if t is not None and 0 < len(t) <= max_tok - 2:\n            out[key] = t\n    return out\n\n\ndef encode_tokens(token_lists, stoi, max_tok=MAX_TOK):\n    arr = np.full((len(token_lists), max_tok), PAD, np.int16)\n    for i, t in enumerate(token_lists):\n        ids = [BOS] + [stoi.get(x, UNK) for x in t] + [EOS]\n        arr[i, :len(ids)] = ids\n    return arr\n\n\n# ---------------------------------------------------------------- spectra -> arrays\ndef spectra_to_arrays(mz_list, int_list, precursor, adducts, ces):\n    """Peak tensors (N, 64, 3), masks (N, 64), metadata (N, 12), neutral masses (N,)."""\n    n = len(mz_list)\n    peaks = np.zeros((n, MAX_PEAKS, 3), np.float32)\n    mask = np.zeros((n, MAX_PEAKS), bool)\n    meta = np.zeros((n, 12), np.float32)\n    mass = np.zeros(n, np.float32)\n    for i in range(n):\n        peaks[i], mask[i] = peak_tensor(mz_list[i], int_list[i], precursor[i], MAX_PEAKS)\n        meta[i] = metadata_vector(adducts[i], ces[i], precursor[i])\n        m = neutral_mass(precursor[i], adducts[i])\n        mass[i] = m if np.isfinite(m) else precursor[i]\n    return peaks, mask, meta, mass\n\n\n# ---------------------------------------------------------------- model\nclass Fourier(nn.Module):\n    """sin/cos features of a scalar (m/z or mass) at log-spaced periods from 0.05 Da to 2000 Da,\n    so the network can read masses to a fraction of a Da."""\n\n    def __init__(self, n_freq=32, min_period=0.05, max_period=2000.0):\n        super().__init__()\n        periods = torch.logspace(math.log10(min_period), math.log10(max_period), n_freq)\n        self.register_buffer("w", 2 * math.pi / periods)\n        self.dim = 2 * n_freq\n\n    def forward(self, x):\n        a = x.unsqueeze(-1) * self.w\n        return torch.cat([a.sin(), a.cos()], -1)\n\n\nclass Spec2Selfies(nn.Module):\n    def __init__(self, vocab_size, d=256, heads=8, layers=4, ff=1024, drop=0.1):\n        super().__init__()\n        self.d = d\n        self.four = Fourier()\n        self.peak_in = nn.Linear(2 * self.four.dim + 1, d)\n        self.meta_in = nn.Linear(12 + self.four.dim, d)\n        enc_layer = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)\n        self.enc = nn.TransformerEncoder(enc_layer, layers, norm=nn.LayerNorm(d), enable_nested_tensor=False)\n        dec_layer = nn.TransformerDecoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)\n        self.dec = nn.TransformerDecoder(dec_layer, layers, norm=nn.LayerNorm(d))\n        self.tok = nn.Embedding(vocab_size, d, padding_idx=PAD)\n        self.pos = nn.Embedding(MAX_TOK, d)\n        self.out = nn.Linear(d, vocab_size)\n\n    def encode(self, peaks, mask, meta, mass):\n        x = torch.cat([self.four(peaks[..., 0]), self.four(peaks[..., 2]), peaks[..., 1:2]], -1)\n        x = self.peak_in(x)\n        m = self.meta_in(torch.cat([meta, self.four(mass)], -1)).unsqueeze(1)\n        x = torch.cat([m, x], 1)\n        valid = torch.cat([torch.ones_like(mask[:, :1]), mask], 1)\n        return self.enc(x, src_key_padding_mask=~valid), ~valid\n\n    def decode(self, tgt, memory, mem_pad):\n        t = tgt.size(1)\n        causal = torch.triu(torch.ones(t, t, dtype=torch.bool, device=tgt.device), 1)\n        h = self.tok(tgt) * math.sqrt(self.d) + self.pos(torch.arange(t, device=tgt.device))\n        h = self.dec(h, memory, tgt_mask=causal, memory_key_padding_mask=mem_pad)\n        return self.out(h)\n\n    def forward(self, peaks, mask, meta, mass, tgt_in):\n        memory, mem_pad = self.encode(peaks, mask, meta, mass)\n        return self.decode(tgt_in, memory, mem_pad)\n\n\n@torch.no_grad()\ndef beam_search(model, peaks, mask, meta, mass, beam=25, max_len=MAX_TOK - 1):\n    """Plain beam search (sum of log-probs). Returns token ids (S, beam, T) and summed log-probs\n    (S, beam). Finished beams are padded with <pad> and keep their score."""\n    memory, mem_pad = model.encode(peaks, mask, meta, mass)\n    s = memory.size(0)\n    memory = memory.repeat_interleave(beam, 0)\n    mem_pad = mem_pad.repeat_interleave(beam, 0)\n    dev = memory.device\n    seqs = torch.full((s * beam, 1), BOS, dtype=torch.long, device=dev)\n    scores = torch.zeros(s, beam, device=dev)\n    scores[:, 1:] = -1e9                     # only one live beam at the start\n    done = torch.zeros(s * beam, dtype=torch.bool, device=dev)\n    base = (torch.arange(s, device=dev) * beam).unsqueeze(1)\n    for _ in range(max_len):\n        logp = F.log_softmax(model.decode(seqs, memory, mem_pad)[:, -1].float(), -1)\n        logp[:, PAD] = -1e9\n        logp[:, BOS] = -1e9\n        logp[done] = -1e9\n        logp[done, PAD] = 0.0                # a finished beam only extends with <pad>, at no cost\n        cand = (scores.view(-1, 1) + logp).view(s, -1)\n        top, idx = cand.topk(beam, dim=1)\n        v = logp.size(1)\n        sel = (base + idx // v).view(-1)\n        tok = (idx % v).view(-1)\n        seqs = torch.cat([seqs[sel], tok.unsqueeze(1)], 1)\n        done = done[sel] | (tok == EOS)\n        scores = top\n        if bool(done.all()):\n            break\n    return seqs[:, 1:].view(s, beam, -1).cpu().numpy(), scores.cpu().numpy()\n\n\n# ---------------------------------------------------------------- training\ndef save_ckpt(path, model, itos, cfg, extra=None):\n    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)\n    torch.save({"model": model.state_dict(), "itos": itos, "cfg": cfg, **(extra or {})}, path)\n\n\ndef load_ckpt(path, device):\n    ck = torch.load(path, map_location="cpu", weights_only=False)\n    cfg = ck["cfg"]\n    model = Spec2Selfies(len(ck["itos"]), d=cfg["d"], heads=cfg["heads"], layers=cfg["layers"],\n                         ff=cfg["ff"], drop=cfg["drop"])\n    model.load_state_dict(ck["model"])\n    return model.to(device).eval(), ck["itos"]\n\n\ndef train(args):\n    from src.data.dataset import load_training, split_by_structure\n\n    torch.manual_seed(SEED)\n    np.random.seed(SEED)\n    dev = pick_device()\n    t0 = time.time()\n    df = load_training(train_path=args.train_path, max_per_structure=args.max_per_structure,\n                       sample=args.sample, seed=SEED)\n    print(f"loaded {len(df):,} spectra / {df.inchikey14.nunique():,} structures in {time.time() - t0:.0f}s", flush=True)\n\n    tok_map = tokenize_structures(df)\n    df = df[df.inchikey14.isin(tok_map)].reset_index(drop=True)\n    itos = SPECIALS + sorted({t for ts in tok_map.values() for t in ts})\n    stoi = {t: i for i, t in enumerate(itos)}\n    print(f"{len(tok_map):,} structures tokenised, vocab {len(itos)}, {len(df):,} spectra kept "\n          f"({time.time() - t0:.0f}s)", flush=True)\n\n    tr, va = split_by_structure(df, val_frac=args.val_frac, seed=SEED)\n\n    def pack(d):\n        peaks, mask, meta, mass = spectra_to_arrays(\n            d.mz.values, d.intensity.values, d.precursor_mz.values, d.adduct.values, d.collision_energy.values)\n        tgt = encode_tokens([tok_map[k] for k in d.inchikey14.values], stoi)\n        return [torch.from_numpy(a) for a in (peaks, mask, meta, mass, tgt)]\n\n    TR, VA = pack(tr), pack(va)\n    print(f"train {len(tr):,} spectra, val {len(va):,} spectra ({time.time() - t0:.0f}s)", flush=True)\n\n    cfg = dict(d=args.d, heads=args.heads, layers=args.layers, ff=args.ff, drop=args.drop)\n    model = Spec2Selfies(len(itos), **cfg).to(dev)\n    n_params = sum(p.numel() for p in model.parameters())\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.98), weight_decay=0.01)\n    steps_per_epoch = math.ceil(len(tr) / args.batch)\n    total = steps_per_epoch * args.epochs\n    warm = min(args.warmup, max(total // 10, 1))\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: min((s + 1) / warm, 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(s / total, 1.0)))))\n    use_amp = dev.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n    print(f"device {dev}, {n_params / 1e6:.1f}M params, {steps_per_epoch} steps/epoch, "\n          f"batch {args.batch}, lr {args.lr}", flush=True)\n\n    def batch_loss(T, ix):\n        peaks, mask, meta, mass, tgt = (x[ix].to(dev) for x in T)\n        tgt = tgt.long()\n        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):\n            logits = model(peaks, mask, meta, mass, tgt[:, :-1])\n        gold = tgt[:, 1:]\n        loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), gold.reshape(-1),\n                               ignore_index=PAD, label_smoothing=args.label_smoothing)\n        keep = gold != PAD\n        acc = ((logits.argmax(-1) == gold) & keep).sum().item() / keep.sum().item()\n        return loss, acc\n\n    @torch.no_grad()\n    def validate():\n        model.eval()\n        ls, ac, n = 0.0, 0.0, 0\n        for i in range(0, len(VA[0]), 512):\n            ix = torch.arange(i, min(i + 512, len(VA[0])))\n            loss, acc = batch_loss(VA, ix)\n            ls += loss.item() * len(ix)\n            ac += acc * len(ix)\n            n += len(ix)\n        model.train()\n        return ls / n, ac / n\n\n    history, best, bad = [], 1e9, 0\n    model.train()\n    for ep in range(args.epochs):\n        perm = torch.randperm(len(TR[0]))\n        run_l, run_a, run_n, te = 0.0, 0.0, 0, time.time()\n        for b in range(steps_per_epoch):\n            ix = perm[b * args.batch:(b + 1) * args.batch]\n            loss, acc = batch_loss(TR, ix)\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n            run_l += loss.item() * len(ix)\n            run_a += acc * len(ix)\n            run_n += len(ix)\n            if (b + 1) % args.log_every == 0:\n                print(f"  ep {ep + 1} step {b + 1}/{steps_per_epoch} loss {run_l / run_n:.4f} "\n                      f"acc {run_a / run_n:.3f} {time.time() - te:.0f}s", flush=True)\n        vl, vacc = validate()\n        row = dict(epoch=ep + 1, train_loss=run_l / run_n, train_token_acc=run_a / run_n,\n                   val_loss=vl, val_token_acc=vacc, epoch_seconds=time.time() - te,\n                   lr=sched.get_last_lr()[0])\n        history.append(row)\n        print(json.dumps(row), flush=True)\n        if vl < best:\n            best, bad = vl, 0\n            save_ckpt(args.out, model, itos, cfg, {"epoch": ep + 1, "val_loss": vl})\n        else:\n            bad += 1\n            if bad >= args.patience:\n                print("early stopping", flush=True)\n                break\n    info = dict(args=vars(args), n_train_spectra=len(tr), n_val_spectra=len(va), n_structures=len(tok_map),\n                vocab_size=len(itos), n_params=n_params, device=str(dev), history=history,\n                best_val_loss=best, total_seconds=time.time() - t0)\n    hist_path = args.history or os.path.join(HERE, "outputs", "E702_train_history.json")\n    os.makedirs(os.path.dirname(hist_path), exist_ok=True)\n    with open(hist_path, "w") as f:\n        json.dump(info, f, indent=1)\n    print(f"best val loss {best:.4f}; checkpoint {args.out}; history {hist_path}", flush=True)\n\n\n# ---------------------------------------------------------------- scorer\ndef query_mass(spectra):\n    masses = [neutral_mass(s["precursor_mz"], s["adduct"]) for s in spectra]\n    masses = [m for m in masses if np.isfinite(m)]\n    return float(np.median(masses)) if masses else np.nan\n\n\nclass SmilesTransformerScorer(Scorer):\n    name = "spec2selfies"\n\n    def __init__(self, weights, beam=25, ppm=15.0, max_spectra=8, max_products=50,\n                 gen_weight=1.0, gate=None, gated_weight=0.3, device=None):\n        self.dev = device or pick_device()\n        self.model, self.itos = load_ckpt(weights, self.dev)\n        self.beam, self.ppm = beam, ppm\n        self.max_spectra, self.max_products = max_spectra, max_products\n        self.gen_weight, self.gate, self.gated_weight = gen_weight, gate, gated_weight\n        self._gen = {}\n        self.stats = []\n\n    def extra_candidates(self, spectra):\n        from rdkit.Chem import MolFromSmiles\n        from rdkit.Chem.rdMolDescriptors import CalcExactMolWt\n\n        t0 = time.time()\n        self._gen = {}\n        st = {"n_spectra": 0, "n_beams": 0, "n_valid": 0, "n_in_mass": 0, "n_unique_valid": 0,\n              "n_unique": 0}\n        M = query_mass(spectra)\n        if not np.isfinite(M):\n            self.stats.append({**st, "keys": [], "valid": [], "seconds": 0.0})\n            return []\n        use = spectra[: self.max_spectra]\n        peaks, mask, meta, mass = spectra_to_arrays(\n            [s["mz"] for s in use], [s["intensity"] for s in use], [s["precursor_mz"] for s in use],\n            [s["adduct"] for s in use], [s["collision_energy"] for s in use])\n        mass[:] = M                                    # every spectrum of a molecule shares one M\n        ids, logp = beam_search(\n            self.model, torch.from_numpy(peaks).to(self.dev), torch.from_numpy(mask).to(self.dev),\n            torch.from_numpy(meta).to(self.dev), torch.from_numpy(mass).to(self.dev), beam=self.beam)\n        st["n_spectra"] = len(use)\n        agg, smi_of, valid_all = {}, {}, {}\n        for si in range(ids.shape[0]):\n            best_here = {}\n            for bi in range(ids.shape[1]):\n                toks = []\n                for t in ids[si, bi]:\n                    if t == EOS or t == PAD:\n                        break\n                    toks.append(self.itos[t])\n                st["n_beams"] += 1\n                smi = tokens_to_smiles(toks) if toks else None\n                if smi is None:\n                    continue\n                st["n_valid"] += 1\n                key = smiles_to_ik14(smi, tautomer=False)\n                if not key:\n                    continue\n                valid_all[key] = smi\n                if abs(CalcExactMolWt(MolFromSmiles(smi)) - M) > M * self.ppm * 1e-6:\n                    continue\n                st["n_in_mass"] += 1\n                s = math.exp(float(logp[si, bi]) / (len(toks) + 1))     # +1: the <eos> token\n                if s > best_here.get(key, 0.0):\n                    best_here[key] = s\n                smi_of[key] = smi\n            for key, s in best_here.items():\n                agg[key] = agg.get(key, 0.0) + s / len(use)\n        ranked = sorted(agg.items(), key=lambda kv: -kv[1])[: self.max_products]\n        self._gen = dict(ranked)\n        st["n_unique_valid"] = len(valid_all)\n        st["n_unique"] = len(ranked)\n        self.stats.append({**st, "keys": [k for k, _ in ranked], "valid": list(valid_all.values()),\n                           "seconds": time.time() - t0})\n        return [smi_of[k] for k, _ in ranked]\n\n    def score(self, spectra, candidates):\n        scores = candidates.cosine.values.astype(float).copy()\n        gen = candidates.struct_idx.values == -1\n        w = self.gen_weight\n        if self.gate is not None and (~gen).any() and scores[~gen].max() >= self.gate:\n            w = self.gated_weight\n        scores[gen] = [w * self._gen.get(k, 0.0) for k in candidates.inchikey14.values[gen]]\n        return scores\n\n\ndef summarise(res, model, structs_path="data/processed/structures.parquet"):\n    """Generation diagnostics for the failure analysis, joined to evaluate()\'s per-molecule table.\n\n    The true structure of each validation molecule is looked up by InChIKey14 in the shared\n    processed library table, only to measure how close the generated structures are (formula,\n    similarity). It is never used for training or for ranking."""\n    import pandas as pd\n    from rdkit import Chem, DataStructs, RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n\n    RDLogger.DisableLog("rdApp.*")\n    structs = pd.read_parquet(structs_path).set_index("inchikey14")\n    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\n    per = res["per_molecule"].copy()\n    st = model.stats[: len(per)]\n    for col in ["n_spectra", "n_beams", "n_valid", "n_in_mass", "n_unique_valid", "n_unique", "seconds"]:\n        per[col] = [s[col] for s in st]\n    true_gen, formula_hit, best_tan, mass_hit = [], [], [], []\n    for mol_key, s in zip(per.molecule, st):\n        true_smi = structs.smiles.get(mol_key)\n        tm = Chem.MolFromSmiles(true_smi) if true_smi else None\n        true_gen.append(mol_key in set(s["keys"]))\n        if tm is None:\n            formula_hit.append(False)\n            best_tan.append(np.nan)\n            mass_hit.append(False)\n            continue\n        tf = CalcMolFormula(tm)\n        tfp = gen.GetFingerprint(tm)\n        fh, bt = False, 0.0\n        for smi in s["valid"]:\n            m = Chem.MolFromSmiles(smi)\n            if m is None:\n                continue\n            if CalcMolFormula(m) == tf:\n                fh = True\n            bt = max(bt, DataStructs.TanimotoSimilarity(tfp, gen.GetFingerprint(m)))\n        formula_hit.append(fh)\n        best_tan.append(bt if s["valid"] else np.nan)\n        mass_hit.append(s["n_in_mass"] > 0)\n    per["true_generated"] = true_gen\n    per["formula_match_any"] = formula_hit\n    per["any_in_mass"] = mass_hit\n    per["best_tanimoto_to_true"] = best_tan\n    summary = {\n        "model": res["model"], "mrr": res["mrr"], "top1": res["top1"], "top10": res["top10"],\n        "top25": res["top25"], "n_molecules": len(per),\n        "mean_beams": float(per.n_beams.mean()),\n        "valid_rate": float(per.n_valid.sum() / max(per.n_beams.sum(), 1)),\n        "in_mass_rate_of_valid": float(per.n_in_mass.sum() / max(per.n_valid.sum(), 1)),\n        "in_mass_rate_of_beams": float(per.n_in_mass.sum() / max(per.n_beams.sum(), 1)),\n        "mean_unique_in_mass": float(per.n_unique.mean()),\n        "mol_with_any_in_mass": float(per.any_in_mass.mean()),\n        "mol_with_formula_match": float(per.formula_match_any.mean()),\n        "recall_generated": float(per.true_generated.mean()),\n        "mean_best_tanimoto": float(np.nanmean(per.best_tanimoto_to_true)),\n        "sec_per_molecule": float(per.seconds.mean()),\n    }\n    return per, summary\n\n\ndef evaluate_cli(args):\n    from src.evaluation.validate import evaluate\n\n    torch.manual_seed(SEED)\n    np.random.seed(SEED)\n    model = SmilesTransformerScorer(args.weights, beam=args.beam, gen_weight=args.gen_weight,\n                                    gate=args.gate, max_spectra=args.max_spectra)\n    res = evaluate(model, hide_true=args.hide_true, limit=args.limit)\n    per, summary = summarise(res, model)\n    summary.update(hide_true=args.hide_true, gen_weight=args.gen_weight, gate=args.gate,\n                   beam=args.beam, max_spectra=args.max_spectra, limit=args.limit,\n                   weights=os.path.basename(args.weights))\n    print(json.dumps(summary, indent=1))\n    if args.tag:\n        out = os.path.join(HERE, "outputs")\n        os.makedirs(out, exist_ok=True)\n        per.to_csv(os.path.join(out, f"{args.tag}_per_molecule.csv"), index=False)\n        with open(os.path.join(out, f"{args.tag}_summary.json"), "w") as f:\n            json.dump(summary, f, indent=1)\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    sub = ap.add_subparsers(dest="cmd", required=True)\n    t = sub.add_parser("train")\n    t.add_argument("--train_path", default="data/raw/train.parquet")\n    t.add_argument("--max_per_structure", type=int, default=2)\n    t.add_argument("--sample", type=int, default=None)\n    t.add_argument("--epochs", type=int, default=10)\n    t.add_argument("--batch", type=int, default=256)\n    t.add_argument("--lr", type=float, default=5e-4)\n    t.add_argument("--warmup", type=int, default=1000)\n    t.add_argument("--val_frac", type=float, default=0.05)\n    t.add_argument("--patience", type=int, default=3)\n    t.add_argument("--label_smoothing", type=float, default=0.0)\n    t.add_argument("--d", type=int, default=256)\n    t.add_argument("--heads", type=int, default=8)\n    t.add_argument("--layers", type=int, default=4)\n    t.add_argument("--ff", type=int, default=1024)\n    t.add_argument("--drop", type=float, default=0.1)\n    t.add_argument("--log_every", type=int, default=100)\n    t.add_argument("--out", default=os.path.join(HERE, "weights", "model.pt"))\n    t.add_argument("--history", default=None)\n    e = sub.add_parser("evaluate")\n    e.add_argument("--weights", default=os.path.join(HERE, "weights", "model.pt"))\n    e.add_argument("--hide_true", action="store_true")\n    e.add_argument("--limit", type=int, default=None)\n    e.add_argument("--beam", type=int, default=25)\n    e.add_argument("--max_spectra", type=int, default=8)\n    e.add_argument("--gen_weight", type=float, default=1.0)\n    e.add_argument("--gate", type=float, default=None)\n    e.add_argument("--tag", default=None, help="experiment ID, e.g. E702; saves outputs")\n    a = ap.parse_args()\n    train(a) if a.cmd == "train" else evaluate_cli(a)\n',
}
for path, text in FILES.items():
    full = os.path.join(REPO, path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    open(full, 'w').write(text)
sys.path.insert(0, REPO)
os.chdir(REPO)
print(f'wrote {len(FILES)} files to {REPO}')

In [ ]:
try:
    import rdkit
except ImportError:
    # Kaggle's image has no RDKit. Install the wheels from the private dataset casmi-yash-wheels.
    WHEELS = glob.glob("/kaggle/input/**/rdkit-*.whl", recursive=True)
    if not WHEELS:
        raise RuntimeError("RDKit wheel not found: attach the dataset casmi-yash-wheels")
    os.system(f"pip install --no-index --find-links {os.path.dirname(WHEELS[0])} rdkit pillow -q")
    import rdkit
print("rdkit", rdkit.__version__)

TEST = glob.glob("/kaggle/input/**/test.parquet", recursive=True)[0]
TRAIN = glob.glob("/kaggle/input/**/train.parquet", recursive=True)[0]
DATA = os.path.dirname(TEST)
print("data:", DATA)


In [ ]:
from src.data.build_reference import build

REF = "/kaggle/working/ref"
build(TRAIN, REF)                                   # ~3-5 min
print(f"reference built in {time.time() - t0:.0f}s")


In [ ]:
from members.YashShevkar30.model_analog_enum import AnalogEnumScorer
model = AnalogEnumScorer(ref_dir='/kaggle/working/ref', exclude_libs=(), gate=0.7)   # full library

In [ ]:
from src.evaluation.validate import make_submission

sub = make_submission(model, TEST, REF, "/kaggle/working/submission.csv")
for f in os.listdir(REF):                           # keep only submission.csv in the output
    os.remove(os.path.join(REF, f))
os.rmdir(REF)
guesses = sub.smiles.str.split(";").map(len)
assert sub.molecule_id.is_unique and not sub.isna().any().any() and guesses.max() <= 25
print(sub.shape, f"guesses per molecule: {guesses.min()} to {guesses.max()}", f"total {time.time() - t0:.0f}s")
sub.head()
